In [68]:
import numpy as np
from random import random, seed

np.random.seed(42)

def rosenbrock(X):
  """
  Função de Rosenbrock

  X -> Vetor de entrada

  """
  X = np.array(X)
  if X.ndim > 1:
    return np.sum(100 * (X[:, 1:] - X[:, :-1]**2)**2 + (X[:, :-1] - 1)**2, axis=1)
  else:
    return np.sum(100 * (X[1:] - X[:-1]**2)**2 + (X[:-1] - 1)**2, axis=0)

In [69]:
import numpy as np


def criar_populacao(dimensao, num_individuos, intervalo):
  """
  Função que cria uma população com N indivíduos.

  dimensao -> número de dimensões do problema
  n_individuos -> número de indivíduos na população
  intervalo -> intervalo de valores possíveis para cada dimensão

  """

  populacao = []
  for i in range(num_individuos):
    individuo = []
    for j in range(dimensao):
      individuo.append(np.random.uniform(intervalo[0], intervalo[1]))
    populacao.append(individuo)

  return np.array(populacao)

In [70]:
def melhor_individuo(populacao, funcao_avaliacao):
  fitness = funcao_avaliacao(populacao)
  melhor_individuo = populacao[np.argmin(fitness)]
  return melhor_individuo

In [71]:
def mutacao(populacao, individuo_base, F=0.9):

  rand1 = np.random.randint(0, len(populacao))
  rand2 = np.random.randint(0, len(populacao))
  while (rand1 == rand2):
    rand2 = np.random.randint(0, len(populacao))

  individuo_base = np.array(individuo_base)
  populacao = np.array(populacao)
  vetor_mutante = individuo_base + F * (populacao[rand1] - populacao[rand2])

  return vetor_mutante



In [72]:
from random import random
def cruzamento(populacao, vetor_mutante, CR= 0.9):
  x_rand = populacao[np.random.randint(low=0, high=populacao.shape[0])]

  individuo_cruzado = []
  for i in range(len(x_rand)):
    if np.random.random() < CR:
      individuo_cruzado.append(vetor_mutante[i])
    else:
      individuo_cruzado.append(x_rand[i])

  return individuo_cruzado

In [73]:
def selecao(best, teste):
    if rosenbrock(best) <= rosenbrock(teste):
        return best
    else:
        return teste

In [74]:
pop = criar_populacao(2, 10, [-5, 10])
best = melhor_individuo(pop, rosenbrock)
vet = mutacao(pop, best)

In [75]:
pop

array([[ 0.61810178,  9.2607146 ],
       [ 5.97990913,  3.97987726],
       [-2.65972039, -2.66008219],
       [-4.12874582,  7.99264219],
       [ 4.01672518,  5.62108867],
       [-4.69123259,  9.54864778],
       [ 7.48663961, -1.81491334],
       [-2.27262549, -2.24893235],
       [-0.43636636,  2.87134647],
       [ 1.47917528, -0.6315629 ]])

In [104]:
def algoritmo_evolutivo(problema=rosenbrock, dimensao=2, n_individuos=100, intervalo=[-5,10], f=0.9, cr= 0.9, parada=100000):
    populacao = criar_populacao(dimensao, n_individuos, intervalo)

    best = melhor_individuo(populacao, problema)

    ite = 0
    while (ite < parada):
        for individuo in populacao:
            if ite >= parada:
                break            
            vetor_mutante = mutacao(populacao, individuo, f)

            teste = cruzamento(populacao, vetor_mutante, cr)

            if problema(teste) < problema(individuo):
                idx = np.where(np.all(populacao == individuo, axis=1))[0][0]
                populacao[idx] = teste
            ite += 2

    best = melhor_individuo(populacao, problema)
    print(f"Melhor individuo: {best}\nF(x*)= {problema(best)}\niterações: {ite}")


In [106]:
algoritmo_evolutivo(dimensao=2)

Melhor individuo: [1. 1.]
F(x*)= 1.0630126172049217e-21
iterações: 100000


## Differential Evolution com critério de parada por avaliações ou tolerância

A seguir está uma versão recriada do algoritmo DE, usando:
- número máximo de avaliações da função objetivo
- parada por tolerância entre gerações


In [78]:
def algoritmo_de(problema=rosenbrock, dimensao=2, n_individuos=100, intervalo=(-5, 10), F=0.8, CR=0.9, max_avaliacoes=5000, epsilon=1e-6):
    populacao = criar_populacao(dimensao, n_individuos, intervalo)
    fitness = np.array([problema(individuo) for individuo in populacao])
    avaliacoes = len(populacao)

    melhor_idx = np.argmin(fitness)
    melhor = populacao[melhor_idx].copy()
    melhor_fitness = fitness[melhor_idx]
    historico = [melhor_fitness]

    limite_inferior, limite_superior = intervalo

    while avaliacoes < max_avaliacoes:
        melhor_fitness_anterior = melhor_fitness
        nova_populacao = populacao.copy()
        nova_fitness = fitness.copy()

        for i in range(n_individuos):
            if avaliacoes >= max_avaliacoes:
                break

            candidatos = [indice for indice in range(n_individuos) if indice != i]
            r1, r2, r3 = np.random.choice(candidatos, 3, replace=False)

            vetor_mutante = populacao[r1] + F * (populacao[r2] - populacao[r3])
            vetor_mutante = np.clip(vetor_mutante, limite_inferior, limite_superior)

            j_rand = np.random.randint(dimensao)
            vetor_teste = np.array([
                vetor_mutante[j] if (np.random.random() < CR or j == j_rand) else populacao[i][j]
                for j in range(dimensao)
            ])

            fitness_teste = problema(vetor_teste)
            avaliacoes += 1

            if fitness_teste <= fitness[i]:
                nova_populacao[i] = vetor_teste
                nova_fitness[i] = fitness_teste

                if fitness_teste < melhor_fitness:
                    melhor_fitness = fitness_teste
                    melhor = vetor_teste.copy()

        populacao = nova_populacao
        fitness = nova_fitness
        historico.append(melhor_fitness)

        if abs(melhor_fitness_anterior - melhor_fitness) < epsilon:
            break

    print(f"Melhor individuo: {melhor}\nF(x*) = {melhor_fitness}\nAvaliações: {avaliacoes}")
    return melhor, melhor_fitness, avaliacoes, historico


In [102]:
melhor, melhor_fitness, avaliacoes, historico = algoritmo_de(dimensao=2)


Melhor individuo: [1.31460225 1.51139627]
F(x*) = 4.7984530825582885
Avaliações: 200
